In [1]:
pip install pyserial requests

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: C:\Users\Benize\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [5]:
import serial
import requests
import time
import json

COM_PORT = "COM5"
BAUD_RATE = 9600

LM_STUDIO_URL = "http://127.0.0.1:1234/v1/chat/completions"

try:
    arduino = serial.Serial(COM_PORT, BAUD_RATE, timeout=1)
    time.sleep(3)
    print("Arduino Connected.")
except Exception as e:
    print("Arduino Error:", e)
    arduino = None

chat_history = []

def read_sensor_data():
    default_data = "Gas: 0 Temp: 0 Tilt: 0"

    if arduino is None:
        return default_data

    try:
        arduino.reset_input_buffer()
        sensor_data = arduino.readline().decode("utf-8", errors="ignore").strip()

        if sensor_data == "":
            return default_data

        return sensor_data

    except Exception as e:
        print("Sensor Error:", e)
        return default_data


def parse_sensor_data(sensor_data):
    result = {"gas": 0, "temp": 0, "tilt": 0}

    try:
        tokens = sensor_data.split()
        result["gas"] = int(tokens[1])
        result["temp"] = float(tokens[3])
        result["tilt"] = int(tokens[5])
    except:
        pass

    return result


def analyze_safety(values):
    warnings = []

    gas = values["gas"]
    temp = values["temp"]
    tilt = values["tilt"]

    if gas >= 400:
        warnings.append("CRITICAL GAS LEAK")
    elif gas >= 250:
        warnings.append("HIGH GAS LEVEL")

    if temp >= 60:
        warnings.append("EXTREME HEAT")
    elif temp >= 45:
        warnings.append("HIGH TEMPERATURE")

    if tilt == 1:
        warnings.append("GAS TANK UNSTABLE")

    if len(warnings) == 0:
        warnings.append("Kitchen conditions are safe.")

    return warnings


def ask_ai(food_input, sensor_data, warnings):
    payload = {
        "messages": [
            {
                "role": "system",
                "content": "You are an AI Kitchen Assistant. Tasks: estimate calories, give cooking advice, analyze safety. Keep answers under 40 words."
            },
            {
                "role": "user",
                "content": f"Food: {food_input}\nSensors:\n{sensor_data}\nWarnings:\n{warnings}"
            }
        ],
        "temperature": 0.6,
        "max_tokens": 80
    }

    try:
        response = requests.post(LM_STUDIO_URL, json=payload, timeout=30)
        data = response.json()

        if "choices" in data:
            return data["choices"][0]["message"]["content"]
        elif "error" in data:
            return f"LM Studio Error: {data['error']}"
        else:
            return "Invalid AI response."

    except Exception as e:
        return f"AI Error: {e}"


def send_to_lcd(message):
    if arduino is None:
        return

    try:
        message = message.replace("\n", " ")
        message = message[:32]

        line1 = message[:16]
        line2 = message[16:32]

        arduino.write((line1 + "\n").encode())
        time.sleep(0.5)
        arduino.write((line2 + "\n").encode())
        time.sleep(0.5)

    except Exception as e:
        print("LCD Error:", e)


def save_history():
    try:
        with open("kitchen_history.json", "w") as file:
            json.dump(chat_history, file, indent=4)
    except Exception as e:
        print("Save Error:", e)


print("=" * 60)
print("AI KITCHEN GUARDIAN STARTED")
print("=" * 60)

while True:
    sensor_data = read_sensor_data()
    print("\nSensor Data:", sensor_data)

    sensor_values = parse_sensor_data(sensor_data)

    warnings = analyze_safety(sensor_values)

    print("\nSafety Analysis:")
    for warning in warnings:
        print("-", warning)

    food_input = input("\nEnter food (example: chicken 200g)\nType QUIT to exit:\n> ").strip()

    if food_input == "":
        continue

    if food_input.upper() == "QUIT":
        break

    print("\nThinking...\n")

    ai_response = ask_ai(food_input, sensor_data, warnings)

    print("=" * 60)
    print("USER:")
    print(food_input)

    print("\nAI RESPONSE:")
    print(ai_response)

    print("=" * 60)

    chat_history.append({
        "user": food_input,
        "sensor_data": sensor_data,
        "warnings": warnings,
        "ai_response": ai_response
    })

    save_history()

    send_to_lcd(ai_response)

if arduino:
    arduino.close()

print("\nAI Kitchen Guardian Stopped.")

Arduino Connected.
AI KITCHEN GUARDIAN STARTED

Sensor Data: Gas: 147 Temp: 30.20 Tilt: 0

Safety Analysis:
- Kitchen conditions are safe.

Thinking...

USER:
Hello!

AI RESPONSE:
Hello! I'm a smart kitchen assistant, but I need more information to estimate calories or give cooking advice. Can you please provide more details about the dish you're preparing?

Sensor Data: Gas: 144 Temp: 30.50 Tilt: 0

Safety Analysis:
- Kitchen conditions are safe.

Thinking...

USER:
Can you convert the calories of rice 200g

AI RESPONSE:
Rice contains approximately 695 calories per 200 grams.

Sensor Data: Gas: 141 Temp: 30.60 Tilt: 0

Safety Analysis:
- Kitchen conditions are safe.

Thinking...

USER:
now provide me hotdogs 150g

AI RESPONSE:
Calories for 150g of hotdogs = 280 kcal (based on typical calorie content in hot dogs).

Cooking advice:
1. Cook hotdogs to an internal temperature of 145°F (63°C).
2. Serve immediately after cooking.
3. Allow the hotdogs to cool before serving.

Safety analysis